# Task 6: House Price Prediction

## Objective
Predict house prices using property features such as size, bedrooms, and location using regression models.

## Approach
- Perform preprocessing on features like square footage, number of bedrooms, location
- Train regression models (Linear Regression & Gradient Boosting)
- Evaluate with Mean Absolute Error (MAE) and RMSE
- Visualize predicted vs actual prices

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print("All libraries imported successfully!")

## 2. Load Dataset
Load the House Price Prediction dataset from Kaggle using credentials from environment variables (`.env` file).

> **Security Note:** Kaggle credentials are loaded from environment variables — never hardcoded. See `.env.example` for the required format.

In [ ]:
import os
from dotenv import load_dotenv
import kagglehub

# Load Kaggle credentials from .env file (not tracked by git)
load_dotenv()
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME', '')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY', '')

if not os.environ['KAGGLE_USERNAME'] or not os.environ['KAGGLE_KEY']:
    print("WARNING: Kaggle credentials not found in .env file.")
    print("Falling back to sklearn California Housing dataset...")
    from sklearn.datasets import fetch_california_housing
    data = fetch_california_housing(as_frame=True)
    df = data.frame
    df.columns = ['median_income', 'house_age', 'avg_rooms', 'avg_bedrooms',
                   'population', 'avg_occupancy', 'latitude', 'longitude', 'house_value']
else:
    print("Downloading House Price dataset from Kaggle...")
    path = kagglehub.dataset_download("yasserh/housing-prices-dataset")
    df = pd.read_csv(f"{path}/Housing.csv")
    print(f"Dataset loaded from: {path}")
    print(f"Columns: {df.columns.tolist()}")

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

### Data Dictionary (California Housing)
- **median_income**: Median income in block group ($10,000s)
- **house_age**: Median house age in block group
- **avg_rooms**: Average number of rooms per household
- **avg_bedrooms**: Average number of bedrooms per household
- **population**: Block group population
- **avg_occupancy**: Average household occupancy
- **latitude**: Block group latitude
- **longitude**: Block group longitude
- **house_value**: Median house value ($100,000s) — **target**

### Data Dictionary (Kaggle Housing.csv, if loaded)
- **price**: House price (target)
- **area**: Square footage
- **bedrooms**: Number of bedrooms
- **bathrooms**: Number of bathrooms
- **stories**: Number of stories
- **mainroad**, **guestroom**, **basement**: Yes/No features
- **airconditioning**, **parking**, **prefarea**: Amenities
- **furnishingstatus**: Furnished/Semi-furnished/Unfurnished

In [ ]:
print("Dataset Info:")
df.info()
print("\nDescriptive Statistics:")
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
# Handle potential column name differences between datasets
target_col = 'house_value' if 'house_value' in df.columns else 'price'
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

# If loaded from Kaggle, encode categorical variables
for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes if not col == target_col else df[col]

In [ ]:
if len(num_cols) <= 9:
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    features_to_plot = [c for c in num_cols if c != target_col][:8]
    for i, feat in enumerate(features_to_plot):
        row, col = i // 3, i % 3
        axes[row, col].scatter(df[feat], df[target_col], alpha=0.3, s=5)
        axes[row, col].set_xlabel(feat.replace('_', ' ').title())
        axes[row, col].set_ylabel('Price')
        axes[row, col].set_title(f'{feat} vs Price')
        axes[row, col].grid(True, alpha=0.3)
    axes[2, 2].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("Too many features for grid plot. Showing correlation heatmap instead.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df[target_col], kde=True, bins=50, ax=axes[0])
axes[0].set_title(f'Distribution of {target_col}')

sns.boxplot(data=df, y=target_col, ax=axes[1])
axes[1].set_title(f'Box Plot of {target_col}')

# Plot against first numeric feature
first_feat = [c for c in num_cols if c != target_col][0]
sns.scatterplot(data=df, x=first_feat, y=target_col, alpha=0.3, ax=axes[2])
axes[2].set_title(f'{first_feat} vs Price')

plt.tight_layout()
plt.show()

print(f"Skewness of target: {df[target_col].skew():.3f}")

In [ ]:
plt.figure(figsize=(12, 10))
correlation = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nTop features correlated with target:")
print(correlation[target_col].abs().sort_values(ascending=False))

## 4. Feature Engineering

In [ ]:
# Create interaction features if relevant columns exist
if 'avg_rooms' in df.columns and 'avg_occupancy' in df.columns:
    df['rooms_per_household'] = df['avg_rooms'] / df['avg_occupancy']
    df['bedrooms_per_room'] = df['avg_bedrooms'] / df['avg_rooms']
    df['population_per_household'] = df['population'] / df['avg_occupancy']
if 'latitude' in df.columns and 'longitude' in df.columns:
    df['location_score'] = (df['latitude'] * df['longitude']).abs()

print("Feature engineering complete!")
print(f"Total features: {len(df.columns)}")

## 5. Preprocessing

In [ ]:
X = df.drop(target_col, axis=1)
y = df[target_col]

# Handle any remaining non-numeric columns
X = X.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Model Training

### 6.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"MAE:  {mae_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R² Score: {r2_lr:.4f}")

### 6.2 Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test)

mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Results:")
print(f"MAE:  {mae_gbr:.4f}")
print(f"RMSE: {rmse_gbr:.4f}")
print(f"R² Score: {r2_gbr:.4f}")

## 7. Model Evaluation

### 7.1 Predicted vs Actual Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (name, y_pred) in zip(axes, [('Linear Regression', y_pred_lr), ('Gradient Boosting', y_pred_gbr)]):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Values')
    ax.set_ylabel('Predicted Values')
    ax.set_title(f'{name}: Predicted vs Actual')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, residuals, y_pred) in zip(axes,
    [('Linear Regression', y_test - y_pred_lr, y_pred_lr),
     ('Gradient Boosting', y_test - y_pred_gbr, y_pred_gbr)]):
    ax.scatter(y_pred, residuals, alpha=0.3, s=10)
    ax.axhline(y=0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Predicted Values')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{name}: Residual Plot')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3 Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, residuals) in zip(axes, [('Linear Regression', y_test - y_pred_lr), ('Gradient Boosting', y_test - y_pred_gbr)]):
    sns.histplot(residuals, kde=True, bins=50, ax=ax)
    ax.axvline(x=0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Prediction Error')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name}: Error Distribution')
plt.tight_layout()
plt.show()

### 7.4 Feature Importance

In [ ]:
if hasattr(gbr, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': gbr.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis_r')
    plt.title('Gradient Boosting - Feature Importance')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

    print("\nTop 5 Most Important Features:")
    print(feature_importance.head(5).to_string(index=False))

## 8. Cross-Validation

In [ ]:
cv_scores_lr = cross_val_score(LinearRegression(), X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_gbr = cross_val_score(GradientBoostingRegressor(n_estimators=100, random_state=42), X_train, y_train, cv=3, scoring='r2')

print(f"Linear Regression - CV R²: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")
print(f"Gradient Boosting - CV R²: {cv_scores_gbr.mean():.4f} (+/- {cv_scores_gbr.std():.4f})")

## 9. Results Summary

### Key Findings:
1. **Best Model**: Gradient Boosting Regressor significantly outperforms Linear Regression
2. **Most Important Features**:
   - Median income / property size: Strongest predictor of price
   - Location features: Geographic location matters
   - Property features: Bedrooms, bathrooms, and amenities

### Conclusion:
The Gradient Boosting model provides the most accurate house price predictions. Property size and location are the dominant factors in determining house prices.

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Gradient Boosting'],
    'MAE': [mae_lr, mae_gbr],
    'RMSE': [rmse_lr, rmse_gbr],
    'R²': [r2_lr, r2_gbr]
})
print(results.to_string(index=False))
print("\nTask 6 completed successfully!")